# Transformer Efficiency Benchmark

This notebook compares two Transformer variants across multiple sequence lengths:

| Variant | Attention type | Memory complexity |
|---------|---------------|------------------|
| **Vanilla** | Standard scaled dot-product (explicit QK^T matrix) | O(T²) |
| **Efficient** | `torch.nn.functional.scaled_dot_product_attention` (Flash / memory-efficient kernel) | O(T) |

Metrics collected per run:
- **FLOPs** – analytical estimate
- **Throughput** (tokens / sec)
- **Peak memory** (MB)
- **Latency** (ms / batch)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.models import load_model
from src.benchmark import run_sweep, results_to_csv

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 1 · Hyperparameters

In [ ]:
D_MODEL    = 512
NUM_HEADS  = 8
NUM_LAYERS = 6
D_FF       = 2048
BATCH_SIZE = 8
SEQ_LENS   = [128, 256, 512, 1024]

# Reduce for quick local runs
WARMUP_STEPS  = 5
MEASURE_STEPS = 20

## 2 · Load models

In [ ]:
variants = {
    'vanilla': load_model('vanilla', d_model=D_MODEL, num_heads=NUM_HEADS,
                          num_layers=NUM_LAYERS, d_ff=D_FF, device=DEVICE),
    'efficient': load_model('efficient', d_model=D_MODEL, num_heads=NUM_HEADS,
                            num_layers=NUM_LAYERS, d_ff=D_FF, device=DEVICE),
}
for name, model in variants.items():
    nparams = sum(p.numel() for p in model.parameters())
    print(f'{name:12s}: {nparams:,} parameters')

## 3 · Run benchmark sweep

In [ ]:
results = run_sweep(
    variants=variants,
    seq_lens=SEQ_LENS,
    batch_size=BATCH_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    warmup_steps=WARMUP_STEPS,
    measure_steps=MEASURE_STEPS,
    device=DEVICE,
    log_wandb=False,
)

## 4 · Results table

In [ ]:
df = pd.DataFrame([
    {
        'Variant': r.variant,
        'Seq Len': r.seq_len,
        'FLOPs': f'{r.flops:.3e}',
        'Tokens/sec': f'{r.tokens_per_sec:,.0f}',
        'Peak Mem (MB)': f'{r.peak_memory_mb:.1f}',
        'Latency (ms)': f'{r.latency_ms:.2f}',
    }
    for r in results
])
df

## 5 · Save results

In [ ]:
results_to_csv(results, '../experiments/results.csv')
print('Saved to experiments/results.csv')

## 6 · Pareto curve — Throughput vs. Peak Memory

Points closer to the **top-left corner** are Pareto-efficient (high throughput, low memory).

In [ ]:
df_plot = pd.DataFrame([
    {
        'variant': r.variant,
        'seq_len': r.seq_len,
        'tokens_per_sec': r.tokens_per_sec,
        'peak_memory_mb': r.peak_memory_mb,
        'flops': r.flops,
    }
    for r in results
])

palette = {'vanilla': '#e07b54', 'efficient': '#4c9be8'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Pareto curve: Throughput vs Memory ---
ax = axes[0]
for variant, grp in df_plot.groupby('variant'):
    grp_sorted = grp.sort_values('seq_len')
    ax.plot(grp_sorted['peak_memory_mb'], grp_sorted['tokens_per_sec'],
            marker='o', label=variant, color=palette[variant], linewidth=2)
    for _, row in grp_sorted.iterrows():
        ax.annotate(f'T={int(row.seq_len)}',
                    (row['peak_memory_mb'], row['tokens_per_sec']),
                    textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.set_xlabel('Peak Memory (MB)')
ax.set_ylabel('Throughput (tokens/sec)')
ax.set_title('Pareto Curve: Throughput vs. Memory')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Latency vs Sequence Length ---
ax2 = axes[1]
for variant, grp in df_plot.groupby('variant'):
    grp_sorted = grp.sort_values('seq_len')
    ax2.plot(grp_sorted['seq_len'].values,
             [r.latency_ms for r in results if r.variant == variant],
             marker='s', label=variant, color=palette[variant], linewidth=2)
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Latency (ms/batch)')
ax2.set_title('Latency vs. Sequence Length')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../experiments/pareto_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to experiments/pareto_curve.png')

## 7 · Efficiency ratio summary

In [ ]:
pivot = df_plot.pivot(index='seq_len', columns='variant', values=['tokens_per_sec', 'peak_memory_mb'])
pivot.columns = ['_'.join(c) for c in pivot.columns]
pivot['speedup'] = pivot['tokens_per_sec_efficient'] / pivot['tokens_per_sec_vanilla']
pivot['memory_ratio'] = pivot['peak_memory_mb_efficient'] / pivot['peak_memory_mb_vanilla']
pivot[['speedup', 'memory_ratio']].style.format('{:.2f}').set_caption(
    'Efficient vs Vanilla: speedup (>1 is better) and memory ratio (<1 is better)'
)